# Classificação — Análise Exploratória de Dados (EDA)

**Dataset:** acidentes em rodovias federais registrados pela PRF em 2024. Fonte: [Portal Brasileiro de Dados Abertos](https://dados.gov.br/dataset/acidentes-rodovias-federais).

**Alvo:** `com_vitima_fatal` (binário) — indica se o acidente teve pelo menos uma vítima fatal. É uma classe minoritária (~7% dos casos), o que torna este um problema de classificação desbalanceada.

**Escopo deste notebook:** apenas exploração e entendimento dos dados — qualidade, distribuições, correlações e o desbalanceamento do alvo. Nenhuma transformação, split ou modelagem acontece aqui.

**Próximas etapas** (em notebooks separados, na pasta `notebooks/`):
1. `02_Preprocessamento_Baseline.ipynb` — split treino/teste, pipeline de pré-processamento e baseline
2. `03_Comparacao_Modelos.ipynb` — comparação de modelos com validação cruzada
3. `04_Otimizacao_Optuna.ipynb` — otimização bayesiana de hiperparâmetros com Optuna

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

## 2. Carregar os dados

In [ ]:
# Dataset: acidentes de transito em rodovias federais (PRF, 2024)
# Fonte: https://dados.gov.br/dataset/acidentes-rodovias-federais
CAMINHO_DADOS = '../data/datatran2024.csv'

df = pd.read_csv(CAMINHO_DADOS, sep=';', encoding='latin1')
print('Linhas x Colunas:', df.shape)
df.head()

### 2.1 Binarização do alvo

A coluna original `classificacao_acidente` tem 3 classes ("Sem Vítimas", "Com Vítimas Feridas", "Com Vítimas Fatais"). Para um problema de classificação binária desbalanceada, criamos `com_vitima_fatal` (1 = houve vítima fatal, 0 = não houve).

Essa recodificação é determinística (não usa estatísticas dos dados), então pode ser feita antes do split sem causar data leakage.

In [ ]:
# Classe alvo binaria: houve vitima fatal no acidente?
df['com_vitima_fatal'] = (df['classificacao_acidente'] == 'Com Vítimas Fatais').astype(int)

print(df['com_vitima_fatal'].value_counts())
print()
print((df['com_vitima_fatal'].value_counts(normalize=True) * 100).round(2).astype(str) + '%')

## 3. Visão geral e qualidade dos dados

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

In [ ]:
# Valores nulos por coluna
nulos = df.isnull().sum().sort_values(ascending=False)
nulos[nulos > 0]

In [ ]:
# Linhas duplicadas
print('Duplicadas:', df.duplicated().sum())

## 4. Distribuições

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
df[num_cols].hist(figsize=(14, 10), bins=30)
plt.tight_layout(); plt.show()

## 5. Correlações

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Matriz de correlação'); plt.show()

## 6. Variável alvo (classe)

In [ ]:
ALVO = 'com_vitima_fatal'  # 1 = houve vitima fatal, 0 = nao houve
df[ALVO].value_counts(normalize=True).plot(kind='bar')
plt.title('Distribuição das classes'); plt.show()

## 7. Conclusão da EDA

- Dataset com 73.156 acidentes, poucos nulos, alvo `com_vitima_fatal` fortemente desbalanceado (~7% positivos).
- Pré-processamento, split, baseline, comparação de modelos e otimização continuam em `02_Preprocessamento_Baseline.ipynb`, `03_Comparacao_Modelos.ipynb` e `04_Otimizacao_Optuna.ipynb`.